In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window


import os
import sys

project_pth = os.path.join(os.getcwd(), '..', '..')
sys.path.append(project_pth)

In [0]:
project_pth

In [0]:
from utils.transformations import reusable


In [0]:
df=spark.read.format("parquet")\
 .load("abfss://bronze@priyaazuredatalake.dfs.core.windows.net/DimUser")

In [0]:
display(df)

Autoloader

In [0]:
df_user = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option(
        "cloudFiles.schemaLocation",
        "abfss://silver@priyaazuredatalake.dfs.core.windows.net/DimUser/Checkpoint"
    )\
    .option("schemaEvolutionMode", "addNewColumns")\
    .load(
        "abfss://bronze@priyaazuredatalake.dfs.core.windows.net/DimUser"
    )

In [0]:
df_user = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option(
        "cloudFiles.schemaLocation",
        "abfss://silver@priyaazuredatalake.dfs.core.windows.net/DimUser/schema"
    ) \
    .option("schemaEvolutionMode", "addNewColumns") \
    .load(
        "abfss://bronze@priyaazuredatalake.dfs.core.windows.net/DimUser"
    )

In [0]:
df_user = reusable().dropColumns(
    df_user,
    ["_rescued_data"]
)

In [0]:
query_user = df_user.writeStream.format("delta") \
    .outputMode("append") \
    .option(
        "checkpointLocation",
        "abfss://silver@priyaazuredatalake.dfs.core.windows.net/DimUser/checkpoint"
    ) \
    .trigger(once=True) \
    .option(
        "path",
        "abfss://silver@priyaazuredatalake.dfs.core.windows.net/DimUser/data"
    ) \
    .toTable("spotify_cata.silver.DimUser")

In [0]:
%sql

SELECT COUNT(*)
FROM spotify_cata.silver.DimUser;

In [0]:
from pyspark.sql.functions import upper, col

df_user = df_user.withColumn(
    "user_name",
    upper(col("user_name"))
)

In [0]:
display(spark.read.format("parquet").load("abfss://bronze@priyaazuredatalake.dfs.core.windows.net/DimUser").withColumn("user_name", upper(col("user_name"))))

In [0]:
import sys

sys.path.append("../../")

from utils.transformations import reusable

In [0]:
display(
    spark.read.format("parquet")
    .load("abfss://bronze@priyaazuredatalake.dfs.core.windows.net/DimUser")
    .withColumn(
        "user_name",
        upper(col("user_name"))
    )
)

In [0]:
df_user_obj = reusable()

In [0]:
df_user_obj = reusable()

df_user = df_user_obj.dropColumns(df_user, ['_rescued_data'])
df_user = df_user.dropDuplicates(['user_id'])

query = df_user.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "abfss://silver@priyaazuredatalake.dfs.core.windows.net/DimUser/DisplayCheckpoint") 


In [0]:
display(spark.read.format("parquet").load("abfss://bronze@priyaazuredatalake.dfs.core.windows.net/DimUser").withColumn("user_name", upper(col("user_name"))))

In [0]:
query_user = df_user.writeStream.format("delta") \
    .outputMode("append") \
    .option(
        "checkpointLocation",
        "abfss://silver@priyaazuredatalake.dfs.core.windows.net/DimUser/checkpoint_clean"
    ) \
    .trigger(once=True) \
    .option(
        "path",
        "abfss://silver@priyaazuredatalake.dfs.core.windows.net/DimUser/data"
    ) \
    .toTable("spotify_cata.silver.DimUser")

DimArtist

In [0]:
df_art = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option(
        "cloudFiles.schemaLocation",
        "abfss://silver@priyaazuredatalake.dfs.core.windows.net/DimArt/schema"
    ) \
    .option("schemaEvolutionMode", "addNewColumns") \
    .load(
        "abfss://bronze@priyaazuredatalake.dfs.core.windows.net/DimArtist"
    )

In [0]:
df_art_display = spark.read.format("parquet") \
    .load("abfss://bronze@priyaazuredatalake.dfs.core.windows.net/DimArtist")

display(df_art_display)

In [0]:
df_art_obj = reusable()

df_art = df_art_obj.dropColumns(
    df_art,
    ["_rescued_data"]
)

df_art = df_art.dropDuplicates(
    ["artist_id"]
)

In [0]:
query_art = df_art.writeStream.format("delta") \
    .outputMode("append") \
    .option(
        "checkpointLocation",
        "abfss://silver@priyaazuredatalake.dfs.core.windows.net/DimArt/checkpoint_clean"
    ) \
    .option("mergeSchema", "true") \
    .trigger(once=True) \
    .option(
        "path",
        "abfss://silver@priyaazuredatalake.dfs.core.windows.net/DimArt/data"
    ) \
    .toTable("spotify_cata.silver.DimArtist")

dimtrack


In [0]:
df_track = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option(
        "cloudFiles.schemaLocation",
        "abfss://silver@priyaazuredatalake.dfs.core.windows.net/DimTrack/schema"
    ) \
    .option("schemaEvolutionMode", "addNewColumns") \
    .load(
        "abfss://bronze@priyaazuredatalake.dfs.core.windows.net/DimTrack"
    )

In [0]:
df_track_display = spark.read.format("parquet") \
    .load("abfss://bronze@priyaazuredatalake.dfs.core.windows.net/DimTrack")

display(df_track_display)

In [0]:
from pyspark.sql.functions import col, when

df_track = df_track.withColumn(
    "durationFlag",
    when(col("duration_sec") < 150, "low")
    .when(col("duration_sec") < 300, "medium")
    .otherwise("high")
)

In [0]:
df_track_obj = reusable()

df_track = df_track_obj.dropColumns(
    df_track,
    ["_rescued_data"]
)

df_track = df_track.dropDuplicates(
    ["track_id"]
)

In [0]:
query_track = df_track.writeStream.format("delta") \
    .outputMode("append") \
    .option(
        "checkpointLocation",
        "abfss://silver@priyaazuredatalake.dfs.core.windows.net/DimTrack/checkpoint_clean"
    ) \
    .option("mergeSchema", "true") \
    .trigger(once=True) \
    .option(
        "path",
        "abfss://silver@priyaazuredatalake.dfs.core.windows.net/DimTrack/data"
    ) \
    .toTable("spotify_cata.silver.DimTrack")

In [0]:
df_track.writeStream.format("delta") \
    .outputMode("append") \
    .option(
        "checkpointLocation",
        "abfss://silver@priyaazuredatalake.dfs.core.windows.net/DimTrack/write_checkpoint"
    ) \
    .trigger(once=True) \
    .option(
        "path",
        "abfss://silver@priyaazuredatalake.dfs.core.windows.net/DimTrack/data"
    ) \
    .toTable("spotify_cata.silver.DimTrack")

In [0]:
df_track_display = spark.read.format("delta") \
    .load("abfss://silver@priyaazuredatalake.dfs.core.windows.net/DimTrack/data")

display(df_track_display)

dim date

In [0]:
df_date = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option(
        "cloudFiles.schemaLocation",
        "abfss://silver@priyaazuredatalake.dfs.core.windows.net/DimDate/schema"
    ) \
    .option("schemaEvolutionMode", "addNewColumns") \
    .load(
        "abfss://bronze@priyaazuredatalake.dfs.core.windows.net/DimDate"
    )

In [0]:
df_date = reusable().dropColumns(
    df_date,
    ["_rescued_data"]
)

query_date = df_date.writeStream.format("delta") \
    .outputMode("append") \
    .option(
        "checkpointLocation",
        "abfss://silver@priyaazuredatalake.dfs.core.windows.net/DimDate/checkpoint_clean"
    ) \
    .trigger(once=True) \
    .option(
        "path",
        "abfss://silver@priyaazuredatalake.dfs.core.windows.net/DimDate/data"
    ) \
    .toTable("spotify_cata.silver.DimDate")

fact stream


In [0]:
df_Factstream = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option(
        "cloudFiles.schemaLocation",
        "abfss://silver@priyaazuredatalake.dfs.core.windows.net/FactStream/schema"
    ) \
    .option("schemaEvolutionMode", "addNewColumns") \
    .load(
        "abfss://bronze@priyaazuredatalake.dfs.core.windows.net/FactStream"
    )

In [0]:
df_Factstream_display = spark.read.format("parquet") \
    .load("abfss://bronze@priyaazuredatalake.dfs.core.windows.net/FactStream")

display(df_Factstream_display)

In [0]:
df_Factstream = reusable().dropColumns(
    df_Factstream,
    ["_rescued_data"]
)

df_Factstream.writeStream.format("delta") \
    .outputMode("append") \
    .option(
        "checkpointLocation",
        "abfss://silver@priyaazuredatalake.dfs.core.windows.net/FactStream/checkpoint"
    ) \
    .trigger(once=True) \
    .option(
        "path",
        "abfss://silver@priyaazuredatalake.dfs.core.windows.net/FactStream/data"
    ) \
    .toTable("spotify_cata.silver.FactStream")

In [0]:
query_factstream = df_Factstream.writeStream.format("delta") \
    .outputMode("append") \
    .option(
        "checkpointLocation",
        "abfss://silver@priyaazuredatalake.dfs.core.windows.net/FactStream/checkpoint_clean"
    ) \
    .trigger(once=True) \
    .option(
        "path",
        "abfss://silver@priyaazuredatalake.dfs.core.windows.net/FactStream/data"
    ) \
    .toTable("spotify_cata.silver.FactStream")

In [0]:
%sql
SELECT COUNT(*)
FROM spotify_cata.silver.FactStream;

In [0]:
df_Factstream_display = spark.read.format("delta") \
    .load("abfss://silver@priyaazuredatalake.dfs.core.windows.net/FactStream/data")

display(df_Factstream_display)